# Self attention & Multi-head attention

## Imports and Setup






In [2]:
import tensorflow as tf
import numpy as np

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import (
    Input,
    Embedding,
    MultiHeadAttention,
    Dense,
    GlobalAveragePooling1D,
)
from tensorflow.keras.models import Model

## 1. Data

### 1.1 Prepare data

In [3]:
sentences = [
"The animal didn't cross the street because it was tired",
"The trophy doesn't fit in the suitcase because it is too large",
"The book was placed on the table because it was heavy",
"The laptop was returned because it was damaged",
"The car stopped because it ran out of fuel"
]

labels = [0,1,2,3,4]

### 1.2 Tokenization

In [9]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)
sequences = tokenizer.texts_to_sequences(sentences)

X = pad_sequences(sequences, padding="post")
y = np.array(labels)

X, X.shape

(array([[ 1,  5,  6,  7,  1,  8,  2,  3,  4,  9,  0,  0],
        [ 1, 10, 11, 12, 13,  1, 14,  2,  3, 15, 16, 17],
        [ 1, 18,  4, 19, 20,  1, 21,  2,  3,  4, 22,  0],
        [ 1, 23,  4, 24,  2,  3,  4, 25,  0,  0,  0,  0],
        [ 1, 26, 27,  2,  3, 28, 29, 30, 31,  0,  0,  0]], dtype=int32),
 (5, 12))

## 2. Build model

### 2.1 Input layer

In [10]:
inputs = Input(shape=(X.shape[1],))

### 2.2 Embedding layer

In [11]:
embedding = Embedding(
    input_dim=len(tokenizer.word_index) + 1,
    output_dim=128,
    input_length=X.shape[1]
)(inputs)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


### 2.3 Multi-head self-attention

In [13]:
attention = MultiHeadAttention(
    num_heads=2,
    key_dim=32
)(embedding, embedding, return_attention_scores=True)

### 2.4 Pooling

In [14]:
pooled = GlobalAveragePooling1D()(attention[0])

### 2.5 Dense layers

In [15]:
dense = Dense(64, activation='relu')(pooled)
outputs = Dense(5, activation='softmax')(dense)

### 2.6 Model

In [16]:
model = Model(inputs=inputs, outputs=outputs)

# Compile
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Summary
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 12)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 12, 128)   │      4,096 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ [(None, 12, 128), │     33,088 │ embedding[0][0],  │
│ (MultiHeadAttentio… │ (None, 2, 12,     │            │ embedding[0][0]   │
│                     │ 12)]              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ multi_head_atten… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │      8,256 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 5)         │        325 │ dense[0][0]       │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 45,765 (178.77 KB)

 Trainable params: 45,765 (178.77 KB)

 Non-trainable params: 0 (0.00 B)

## 3. Train model

In [17]:
model.fit(
    X, y,
    epochs=10,
    verbose=1
)

Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.2000 - loss: 1.6099
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 0.4000 - loss: 1.6068
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.4000 - loss: 1.6045
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.8000 - loss: 1.6017
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.8000 - loss: 1.5982
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.8000 - loss: 1.5942
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.8000 - loss: 1.5895
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 140ms/step - accuracy: 1.0000 - loss: 1.5838
Epoch 9/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 1.0000 - loss: 1.5770
Epoch 10/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 1.0000 - loss: 1.5689


## 4. Prediction

In [22]:
test_sentence = ["The animal didn't cross the street because it was tired"]

seq = tokenizer.texts_to_sequences(test_sentence)
test_data = pad_sequences(seq, padding="post", maxlen=X.shape[1])

model.predict(test_data).argmax()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step


np.int64(0)

In [23]:
# label mapping
label_mapping = {
    0: 'animal',
    1: 'trophy',
    2: 'book',
    3: 'laptop',
    4: 'car'
}

In [27]:
test2 = ["The trophy didn't fit into the brown suitcase because it was too large"]
seq = tokenizer.texts_to_sequences(test2)
test_data = pad_sequences(seq, padding="post", maxlen=X.shape[1])

model.predict(test_data).argmax()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step


np.int64(1)